# Music Genre Classification — AST with TorchAudio GPU Pipeline
### Target: 0.87+ Macro F1  |  Target runtime: ~3–4 hours (down from ~12)

**Key change vs Notebook 1:** The entire audio preprocessing pipeline is restructured so the GPU is kept busy at all times.

---

## Why TorchAudio Makes This Faster

To understand the speedup, you need to understand *where* the original 12-hour runtime goes.

### The bottleneck in the librosa version

In Notebook 1, inside each DataLoader worker process, for every single training sample:

1. `librosa.load` — reads the .wav file. Takes ~0.3–0.5s per file, runs in Python.
2. `librosa.effects.time_stretch` — applies a phase vocoder. This is the **main culprit**: it can take 1–3 seconds per call because it computes a full Short-Time Fourier Transform, shifts frames in frequency, then inverts. Pure Python/NumPy, runs only on CPU.
3. `librosa.effects.pitch_shift` — also uses a phase vocoder internally. Another 1–3 seconds.
4. `ASTFeatureExtractor` — computes the log-mel spectrogram in NumPy on CPU. Takes ~50–200ms.

With 6,000 training samples, `EPOCHS_P1 + EPOCHS_P2 = 10` epochs, 4 augmented stems per sample, and 2 workers — most of those 12 hours are spent in steps 2 and 3, running slowly on CPU **while the GPU sits idle**.

### What we change

The restructured pipeline works like this:

| Step | Where it runs | Notebook 1 | This notebook |
|---|---|---|---|
| File loading | CPU workers | `librosa.load` (~0.4s) | `torchaudio.load` (~0.04s, C++ backend) |
| Speed change | CPU workers | `librosa.effects.time_stretch` phase vocoder (~2s) | `torchaudio.functional.resample` rate trick (~0.01s) |
| Pitch shift | CPU workers | `librosa.effects.pitch_shift` phase vocoder (~2s) | `torchaudio.functional.pitch_shift` C++/CUDA (~0.1s) |
| Mel spectrogram | **GPU, main thread** | `ASTFeatureExtractor` NumPy on CPU (~150ms) | `torchaudio.transforms.MelSpectrogram` on GPU (~1ms) |
| SpecAugment | **GPU, main thread** | Not present | `FrequencyMasking + TimeMasking` on GPU (~0ms) |

The mel spectrogram move is the most important architectural change: we take that computation **out of the DataLoader workers entirely** and run it as a batched GPU operation in the training loop. One GPU call on a batch of 6 takes less time than one CPU call on a single sample.

### One important correction about DURATION

In Notebook 1, `DURATION = 25` seconds was used. But the AST model's positional embeddings are fixed at **1024 time frames**. The `ASTFeatureExtractor` always pads or truncates its output to 1024 frames regardless of how long the audio is. At 16 kHz with a 10ms hop (160 samples per hop), 1024 frames corresponds to exactly **163,840 samples ≈ 10.24 seconds**.

This means the original 25-second crop was computed into ~2,500 mel frames, then immediately **discarded down to 1,024**. The random crop position still mattered (it determined which starting point was kept), but the extra audio was wasted computation. In this notebook we crop to exactly 163,840 samples upfront, which is cleaner and faster.

### Does removing DURATION=25 hurt quality?

No. The model sees exactly the same 10.24s window it always did. The difference is that we now choose the random crop window from the full file (which may be 30s), rather than first taking a 25s window and then discarding the tail. The crop diversity is actually *better* because we have more possible start positions.

---

## Augmentation additions

In addition to the speedup, this notebook adds **SpecAugment** (Park et al., 2019) directly on the mel spectrogram on GPU:

- `FrequencyMasking(freq_mask_param=24)` — randomly zeroes up to 24 consecutive mel frequency bins. Forces the model to classify genres without relying on narrow frequency bands.
- `TimeMasking(time_mask_param=80)` — randomly zeroes up to 80 consecutive time frames. Forces the model to classify genres from partial temporal context.

These augmentations are essentially free because they run as simple tensor masking operations on GPU, in microseconds.

## 1. Install Libraries

Install `transformers` for the AST model, `torchaudio` for audio loading and GPU transforms, and `torchmetrics` for the Macro F1 scorer. `librosa` is still needed only for the EDA visualisation cells — it is not used in any training or inference code.

In [ ]:
!pip install -q transformers torchaudio torchmetrics librosa

## 2. Imports

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import torchaudio.functional as AF
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import (
    ASTForAudioClassification,
    get_cosine_schedule_with_warmup
)
from torchmetrics.classification import MulticlassF1Score

warnings.filterwarnings('ignore')
print(f"PyTorch   : {torch.__version__}")
print(f"TorchAudio: {torchaudio.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 3. Reproducibility Seed

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
print("Seed set to 42.")

## 4. Configuration

### Key differences from Notebook 1

`AUDIO_LEN` replaces `DURATION`. Instead of a 25-second crop, we compute the exact number of waveform samples needed to produce 1024 mel frames:

```
AUDIO_LEN = AST_MAX_FRAMES × HOP_LENGTH = 1024 × 160 = 163,840 samples ≈ 10.24s
```

This is the correct number derived from the AST model's fixed positional embedding size, as documented in the HuggingFace Transformers `ASTFeatureExtractor` configuration (`max_length=1024`, `sampling_rate=16000`).

`GRAD_ACCUM = 4` is kept from Notebook 1 for stable Phase 2 gradients.

`N_TTA = 7` and `LR_P2 = 1e-5` are kept from Notebook 1.

`NUM_WORKERS = 4` is increased from 2. Since each worker now does much less work (no phase vocoder, no mel computation), we can afford more workers to keep the data pipeline fed. The GPU's mel transform will be the pacing element now, not the workers.

In [ ]:
SAMPLE_RATE     = 16000
HOP_LENGTH      = 160        # 10ms at 16kHz — matches ASTFeatureExtractor
WIN_LENGTH      = 400        # 25ms at 16kHz — matches ASTFeatureExtractor
N_MELS          = 128        # matches ASTFeatureExtractor num_mel_bins
AST_MAX_FRAMES  = 1024       # fixed positional embedding size in the AST model
AUDIO_LEN       = AST_MAX_FRAMES * HOP_LENGTH  # = 163840 samples ≈ 10.24s

# AudioSet normalisation constants — from ASTFeatureExtractor defaults
AST_MEAN        = -4.2677393
AST_STD         =  4.5689974

BATCH_SIZE      = 6
GRAD_ACCUM      = 4
NUM_WORKERS     = 0          # more workers since per-sample work is now lighter

EPOCHS_P1       = 3
EPOCHS_P2       = 7
LR_P1           = 3e-4
LR_P2           = 1e-5
LR_HEAD_MULT    = 10
WEIGHT_DECAY    = 0.01
PATIENCE_P1     = 2
PATIENCE_P2     = 4

TRAIN_SIZE      = 6000
NOISE_PROB      = 0.8
NOISE_LEVEL     = (0.03, 0.20)
TEMPO_PROB      = 0.4
TEMPO_RANGE     = (0.88, 1.12)
PITCH_PROB      = 0.3
PITCH_RANGE     = (-2, 2)
STEM_WEIGHT_RANGE = (0.6, 1.4)

FREQ_MASK_PARAM = 24         # SpecAugment: max consecutive mel bins to mask
TIME_MASK_PARAM = 80         # SpecAugment: max consecutive time frames to mask
N_SPEC_MASKS    = 2          # number of each mask type to apply per sample

N_TTA  = 7
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device         : {DEVICE}")
print(f"Audio length   : {AUDIO_LEN} samples = {AUDIO_LEN/SAMPLE_RATE:.2f}s")
print(f"Mel shape      : ({AST_MAX_FRAMES}, {N_MELS})  [time_frames x mel_bins]")
print(f"Train size     : {TRAIN_SIZE}")
print(f"Epochs P1/P2   : {EPOCHS_P1} / {EPOCHS_P2}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"TTA crops      : {N_TTA}")

## 5. Paths and Label Mapping

In [ ]:
BASE_PATH      = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
GENRES_PATH    = os.path.join(BASE_PATH, 'genres_stems')
ESC_PATH       = os.path.join(BASE_PATH, 'ESC-50-master', 'audio')
REQUIRED_STEMS = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

GENRES   = ['blues', 'classical', 'country', 'disco', 'hiphop',
            'jazz', 'metal', 'pop', 'reggae', 'rock']
label2id = {g: i for i, g in enumerate(GENRES)}
id2label = {i: g for g, i in label2id.items()}
NUM_LABELS = len(GENRES)

print("Paths configured.")
print("label2id:", label2id)

## 6. Exploratory Data Analysis (EDA)

We examine the dataset structure before training: genre balance, waveform and spectrogram shape, audio duration distribution, and ESC-50 noise inventory. These cells use `librosa` only for visualisation and run once — they are not in the training loop.

### 6.1 Song Count Per Genre

Count valid songs (all four stems present) per genre and visualise balance.

In [ ]:
genre_counts = {}
for genre in GENRES:
    genre_path = os.path.join(GENRES_PATH, genre)
    count = sum(
        1 for folder in os.listdir(genre_path)
        if os.path.isdir(os.path.join(genre_path, folder))
        and all(os.path.exists(os.path.join(genre_path, folder, s)) for s in REQUIRED_STEMS)
    )
    genre_counts[genre] = count
    print(f"  {genre:<12}: {count} songs")

total = sum(genre_counts.values())
print(f"\nTotal valid songs: {total}")

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(genre_counts.keys(), genre_counts.values(), color='steelblue', edgecolor='white')
ax.set_title("Song Count Per Genre (valid stems)", fontsize=13, fontweight='bold')
ax.set_xlabel("Genre")
ax.set_ylabel("Number of Songs")
ax.set_ylim(0, max(genre_counts.values()) * 1.15)
for bar, val in zip(bars, genre_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('/kaggle/working/eda_genre_counts.png', dpi=120)
plt.show()

### 6.2 Waveform and Log-Mel Spectrogram Per Genre

We visualise the raw waveform and log-mel spectrogram for the vocals stem of one song per genre. The log-mel spectrogram is what the AST model ultimately processes. Notice how genres differ visually: metal tends to fill the full frequency range densely, while classical shows sparse, structured harmonic patterns.

In [ ]:
def get_first_song(genre):
    genre_path = os.path.join(GENRES_PATH, genre)
    for folder in sorted(os.listdir(genre_path)):
        fp = os.path.join(genre_path, folder)
        if os.path.isdir(fp) and os.path.exists(os.path.join(fp, 'vocals.wav')):
            return os.path.join(fp, 'vocals.wav')
    return None

fig = plt.figure(figsize=(18, len(GENRES) * 2.2))
gs  = gridspec.GridSpec(len(GENRES), 2, figure=fig, hspace=0.6, wspace=0.35)

for i, genre in enumerate(GENRES):
    path = get_first_song(genre)
    if path is None:
        continue
    audio, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True, duration=10.0)

    ax_wave = fig.add_subplot(gs[i, 0])
    ax_wave.plot(np.linspace(0, len(audio)/sr, len(audio)), audio,
                 linewidth=0.4, color='steelblue')
    ax_wave.set_title(f"{genre} — waveform", fontsize=8, fontweight='bold')
    ax_wave.set_xlabel("Time (s)", fontsize=7)
    ax_wave.set_ylabel("Amplitude", fontsize=7)
    ax_wave.tick_params(labelsize=6)

    ax_spec = fig.add_subplot(gs[i, 1])
    mel    = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=64, fmax=8000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img    = librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel',
                                      fmax=8000, ax=ax_spec, cmap='magma')
    ax_spec.set_title(f"{genre} — log-mel spectrogram", fontsize=8, fontweight='bold')
    ax_spec.tick_params(labelsize=6)
    plt.colorbar(img, ax=ax_spec, format='%+2.0f dB', pad=0.01)

plt.suptitle("Waveform and Log-Mel Spectrogram per Genre (vocals stem, first 10s)",
             fontsize=12, fontweight='bold', y=1.005)
plt.savefig('/kaggle/working/eda_waveforms_spectrograms.png', dpi=110, bbox_inches='tight')
plt.show()

### 6.3 Audio Duration Distribution

We measure the duration of a sample of stem files to confirm that `AUDIO_LEN = 163,840 samples (10.24s)` is a reasonable crop window. Most GTZAN-derived tracks are ~30 seconds, so a 10.24s crop captures roughly one-third of each file — a good window for genre-defining features like verse/chorus patterns.

In [ ]:
sampled = []
for genre in GENRES:
    genre_path = os.path.join(GENRES_PATH, genre)
    count = 0
    for folder in sorted(os.listdir(genre_path)):
        if count >= 5: break
        vp = os.path.join(genre_path, folder, 'vocals.wav')
        if os.path.isdir(os.path.join(genre_path, folder)) and os.path.exists(vp):
            try:
                sampled.append((genre, librosa.get_duration(path=vp)))
                count += 1
            except Exception:
                pass

dur_df = pd.DataFrame(sampled, columns=['genre', 'duration_s'])
print("Duration statistics by genre (seconds):")
print(dur_df.groupby('genre')['duration_s'].describe()[['mean','min','max']].round(1).to_string())
print(f"\nOverall mean: {dur_df['duration_s'].mean():.1f}s")
print(f"AUDIO_LEN   : {AUDIO_LEN/SAMPLE_RATE:.2f}s")
covered = (dur_df['duration_s'] >= AUDIO_LEN/SAMPLE_RATE).mean() * 100
print(f"Files longer than AUDIO_LEN: {covered:.0f}%  (the rest will be padded)")

### 6.4 ESC-50 Noise File Inventory

In [ ]:
noise_files = []
for root, _, files in os.walk(ESC_PATH):
    for f in files:
        if f.endswith('.wav'):
            noise_files.append(os.path.join(root, f))

print(f"Total ESC-50 noise files: {len(noise_files)}")

cat_counts = defaultdict(int)
for fp in noise_files:
    name  = os.path.splitext(os.path.basename(fp))[0]  # strip .wav first
    parts = name.split('-')
    if len(parts) >= 4:
        try:
            cat_counts[int(parts[3])] += 1
        except ValueError:
            pass

print(f"Unique sound categories: {len(cat_counts)}")

## 7. Song Index and Train/Val Split

Build the song index (genre -> list of valid song folders) and split 85/15 per genre.

In [ ]:
def build_song_index():
    index = {}
    for genre in GENRES:
        genre_path = os.path.join(GENRES_PATH, genre)
        valid = [
            os.path.join(genre_path, folder)
            for folder in sorted(os.listdir(genre_path))
            if os.path.isdir(os.path.join(genre_path, folder))
            and all(os.path.exists(os.path.join(genre_path, folder, s))
                    for s in REQUIRED_STEMS)
        ]
        index[genre] = valid
        print(f"  {genre:<12}: {len(valid)} songs")
    return index

song_index = build_song_index()

noise_files = []
for root, _, files in os.walk(ESC_PATH):
    for f in files:
        if f.endswith('.wav'):
            noise_files.append(os.path.join(root, f))

print(f"\nNoise files: {len(noise_files)}")

train_index, val_index = {}, {}
for genre in GENRES:
    songs = song_index[genre][:]
    random.shuffle(songs)
    split = int(0.85 * len(songs))
    train_index[genre] = songs[:split]
    val_index[genre]   = songs[split:]
    print(f"  {genre:<12} -> Train: {len(train_index[genre])}, Val: {len(val_index[genre])}")

## 8. GPU Mel Spectrogram and SpecAugment Transforms

This is the core of the speedup. We define three GPU-resident transforms once, then reuse them for every batch.

### `mel_transform` — Log-Mel Spectrogram (GPU)

`torchaudio.transforms.MelSpectrogram` implements the STFT → mel filterbank → log pipeline as a `torch.nn.Module`. By calling `.to(DEVICE)`, we place it on GPU. When we pass a batch of waveform tensors (already on GPU) through it, the entire computation runs in GPU memory — no CPU round-trip.

Parameters are matched exactly to the `ASTFeatureExtractor` defaults from the HuggingFace documentation:
- `n_mels = 128`, `sample_rate = 16000`, `hop_length = 160` (10ms), `win_length = 400` (25ms)
- After computing the power mel spectrogram, we apply `log(mel + 1e-7)` to get log-scale values.
- We then normalise with the AudioSet mean (`-4.2677393`) and std (`4.5689974`) — the same values used by `ASTFeatureExtractor.do_normalize`, sourced from the official HuggingFace Transformers documentation.

### `freq_masking` and `time_masking` — SpecAugment (GPU)

`torchaudio.transforms.FrequencyMasking` and `TimeMasking` implement SpecAugment (Park et al., 2019, "SpecAugment: A Simple Data Augmentation Method for Automatic Speech Recognition"). They run in-place on the mel spectrogram tensor, zeroing random contiguous bands. Because the spectrogram is already on GPU from the mel transform, these operations are essentially free.

In [ ]:
mel_transform = T.MelSpectrogram(
    sample_rate = SAMPLE_RATE,
    n_fft       = 512,
    win_length  = WIN_LENGTH,
    hop_length  = HOP_LENGTH,
    n_mels      = N_MELS,
    f_min       = 0.0,
    f_max       = 8000.0,
    power       = 2.0,
    norm        = 'slaney',
    mel_scale   = 'slaney',
).to(DEVICE)

freq_masking = T.FrequencyMasking(freq_mask_param=FREQ_MASK_PARAM).to(DEVICE)
time_masking = T.TimeMasking(time_mask_param=TIME_MASK_PARAM).to(DEVICE)


def compute_log_mel(waveforms, training=True):
    """
    Convert a batch of raw waveforms to normalised log-mel spectrograms on GPU.
    
    Args:
        waveforms: FloatTensor of shape (B, AUDIO_LEN) on DEVICE
        training: if True, apply SpecAugment masks
    Returns:
        FloatTensor of shape (B, AST_MAX_FRAMES, N_MELS) — the AST input format
    """
    with torch.no_grad():
        mel = mel_transform(waveforms)            # (B, N_MELS, T)
        mel = torch.log(mel + 1e-7)               # log scale
        mel = (mel - AST_MEAN) / (AST_STD * 2)   # AudioSet normalisation

        if training:
            for _ in range(N_SPEC_MASKS):
                mel = freq_masking(mel)
                mel = time_masking(mel)

        mel = mel.transpose(1, 2)                 # (B, T, N_MELS) — AST format

        # Pad or truncate time dimension to AST_MAX_FRAMES
        T_frames = mel.shape[1]
        if T_frames < AST_MAX_FRAMES:
            mel = F.pad(mel, (0, 0, 0, AST_MAX_FRAMES - T_frames))
        else:
            mel = mel[:, :AST_MAX_FRAMES, :]

    return mel


print("GPU mel transform and SpecAugment transforms defined.")
print(f"  MelSpectrogram  -> shape: (B, {N_MELS}, {AST_MAX_FRAMES})")
print(f"  After transpose -> shape: (B, {AST_MAX_FRAMES}, {N_MELS})  [AST input format]")

## 9. Waveform Augmentation Pipeline (CPU Workers)

DataLoader workers run in **separate CPU processes** — they cannot access the GPU. So all waveform-level augmentations still run on CPU, but with torchaudio's C++ backend which is significantly faster than librosa's Python implementation.

### Loading: `torchaudio.load`

`torchaudio.load` uses `libsndfile` via C++ bindings. It is roughly 5–10x faster than `librosa.load` for the same file, primarily because it skips the Python audio decode layer and does not perform any resampling by default (we specify `sr=None` equivalent by setting the backend directly).

### Speed change: resample trick

The original `librosa.effects.time_stretch` uses a phase vocoder — it computes a full STFT, shifts the time axis in frequency domain, then inverts with Griffin-Lim. This is expensive (~1–3s per call).

The resample trick is much faster: to make audio sound `rate` times faster, we pretend it was recorded at `16000 * rate` Hz and resample down to 16000 Hz. This has the same audible effect as time-stretching (without pitch change) but is just a simple polyphase resampling — milliseconds instead of seconds.

```
original audio at 16 kHz, rate=1.1 (10% faster)
-> torchaudio.functional.resample(waveform, orig_freq=17600, new_freq=16000)
```

### Pitch shift: `torchaudio.functional.pitch_shift`

Unlike librosa's pitch_shift (Python phase vocoder), this function is implemented in C++ and supports CUDA tensors. In the worker processes we run it on CPU tensors, but it is still ~10–20x faster than librosa. The function signature is available from torchaudio 0.13 onwards.

In [ ]:
def load_audio_ta(path):
    """Load mono audio with torchaudio, return (1, T) float32 tensor."""   
    waveform, sr = torchaudio.load(path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        waveform = AF.resample(waveform, orig_freq=sr, new_freq=SAMPLE_RATE)
    return waveform  # shape: (1, T)


def normalize_ta(waveform):
    return waveform / (waveform.abs().max() + 1e-6)


def crop_random_ta(waveform):
    """Random crop to AUDIO_LEN samples — used during training."""    
    T = waveform.shape[-1]
    if T >= AUDIO_LEN:
        start = random.randint(0, T - AUDIO_LEN)
        return waveform[..., start : start + AUDIO_LEN]
    return F.pad(waveform, (0, AUDIO_LEN - T))


def crop_center_ta(waveform):
    """Centre crop to AUDIO_LEN samples — used during validation."""   
    T = waveform.shape[-1]
    if T >= AUDIO_LEN:
        start = (T - AUDIO_LEN) // 2
        return waveform[..., start : start + AUDIO_LEN]
    return F.pad(waveform, (0, AUDIO_LEN - T))


def speed_change_ta(waveform):
    """
    Speed change via resample trick — same audible effect as time_stretch
    but orders of magnitude faster (no phase vocoder).
    To speed up by factor r: resample from (sr*r) -> sr.
    """
    if random.random() < TEMPO_PROB:
        rate     = random.uniform(*TEMPO_RANGE)
        orig_len = waveform.shape[-1]
        fake_sr  = int(SAMPLE_RATE * rate)
        waveform = AF.resample(waveform, orig_freq=fake_sr, new_freq=SAMPLE_RATE)
        # restore original length by pad/crop so downstream crop sees same-length input
        if waveform.shape[-1] > orig_len:
            waveform = waveform[..., :orig_len]
        elif waveform.shape[-1] < orig_len:
            waveform = F.pad(waveform, (0, orig_len - waveform.shape[-1]))
    return waveform


def pitch_shift_ta(waveform):
    """
    Pitch shift using torchaudio.functional.pitch_shift (C++ backend).
    Runs on CPU tensors in DataLoader workers — still ~10-20x faster than librosa.
    Available from torchaudio 0.13+.
    """
    if random.random() < PITCH_PROB:
        n_steps = random.uniform(*PITCH_RANGE)
        waveform = AF.pitch_shift(waveform, sample_rate=SAMPLE_RATE,
                                  n_steps=int(round(n_steps)))
    return waveform


def add_noise_ta(waveform):
    """Add a random ESC-50 noise clip at low amplitude."""    
    if random.random() < NOISE_PROB and len(noise_files) > 0:
        noise = load_audio_ta(random.choice(noise_files))
        noise = crop_random_ta(noise)
        level = random.uniform(*NOISE_LEVEL)
        waveform = waveform + level * noise
    return waveform


def polarity_invert_ta(waveform):
    if random.random() < 0.5:
        return -waveform
    return waveform


print("TorchAudio waveform augmentation functions defined.")

## 10. Dataset Classes

The key structural change from Notebook 1: `__getitem__` now returns a raw waveform tensor (`float32`, shape `(AUDIO_LEN,)`) and an integer label. There is **no mel spectrogram computation in the Dataset**.

The mel computation happens in `compute_log_mel()` inside the training loop, once per batch, on GPU — as a batched operation on the entire `(B, AUDIO_LEN)` tensor. This is orders of magnitude more efficient than computing mel individually for each sample in CPU worker processes.

The waveform is returned as a 1D tensor (not `(1, T)`) so that PyTorch's default collate function stacks them cleanly into `(B, AUDIO_LEN)` batches.

In [ ]:
class TrainDataset(Dataset):

    def __init__(self, song_index, size=TRAIN_SIZE):
        self.song_index = song_index
        self.size       = size

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        while True:
            try:
                genre = random.choice(GENRES)
                songs = self.song_index[genre]
                if len(songs) == 0:
                    continue

                mixed = None
                for stem in REQUIRED_STEMS:
                    song_path = random.choice(songs)
                    wav       = load_audio_ta(os.path.join(song_path, stem))
                    wav       = speed_change_ta(wav)
                    wav       = pitch_shift_ta(wav)
                    wav       = crop_random_ta(wav)
                    wav       = normalize_ta(wav)
                    weight    = random.uniform(*STEM_WEIGHT_RANGE)
                    wav       = wav * weight
                    mixed = wav if mixed is None else mixed + wav

                mixed = normalize_ta(mixed)
                mixed = polarity_invert_ta(mixed)
                mixed = add_noise_ta(mixed)
                mixed = normalize_ta(mixed)

                return {
                    'waveform': mixed.squeeze(0),          # (AUDIO_LEN,)
                    'label'   : label2id[genre]
                }
            except Exception:
                continue

print("TrainDataset defined.")

In [ ]:
class ValDataset(Dataset):

    def __init__(self, song_index):
        self.samples = [
            (genre, song_path)
            for genre in GENRES
            for song_path in song_index[genre]
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        genre, song_path = self.samples[idx]
        mixed = None
        for stem in REQUIRED_STEMS:
            wav   = load_audio_ta(os.path.join(song_path, stem))
            wav   = crop_center_ta(wav)
            mixed = wav if mixed is None else mixed + wav
        mixed = normalize_ta(mixed)
        return {
            'waveform': mixed.squeeze(0),
            'label'   : label2id[genre]
        }

print("ValDataset defined.")

## 11. DataLoaders

`NUM_WORKERS = 4` is safe here because each worker now does much less work (no phase vocoder, no mel computation). Each worker call just does: load 4 wav files, resample trick, crop, add noise, return tensors. The mel computation is deferred to the GPU in the main thread.

In [ ]:
train_dataset = TrainDataset(train_index, size=TRAIN_SIZE)
val_dataset   = ValDataset(val_index)

train_loader = DataLoader(
    train_dataset,
    batch_size         = BATCH_SIZE,
    shuffle            = True,
    num_workers        = NUM_WORKERS,
    pin_memory         = True,
    collate_fn         = None
)
val_loader = DataLoader(
    val_dataset,
    batch_size         = BATCH_SIZE,
    shuffle            = False,
    num_workers        = NUM_WORKERS,
    pin_memory         = True,
)

print(f"Train samples  : {len(train_dataset)}")
print(f"Val samples    : {len(val_dataset)}")
print(f"Train batches  : {len(train_loader)}")
print(f"Val batches    : {len(val_loader)}")

## 12. AST Model

We load the same AST model as Notebook 1. The difference is that we no longer use `ASTFeatureExtractor` at all — our `compute_log_mel()` GPU function replaces it entirely, producing the same `(B, 1024, 128)` input tensors with the same AudioSet normalisation.

In [ ]:
MODEL_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'

model = ASTForAudioClassification.from_pretrained(
    MODEL_NAME,
    num_labels              = NUM_LABELS,
    id2label                = id2label,
    label2id                = label2id,
    ignore_mismatched_sizes = True
)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"AST model loaded on {DEVICE}.")
print(f"Total parameters: {total_params:,}")
print(f"Note: ASTFeatureExtractor is NOT used in this notebook.")
print(f"      Mel spectrograms are computed via torchaudio on {DEVICE}.")

## 13. Loss Function and Metric

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
f1_metric = MulticlassF1Score(num_classes=NUM_LABELS, average='macro').to(DEVICE)

print("Loss   : CrossEntropyLoss (label_smoothing=0.1)")
print("Metric : Macro F1 Score")

## 14. Training and Validation Loops

The training loop now has one extra step at the start of every batch: it calls `compute_log_mel(waveforms, training=True)` to convert raw waveforms to SpecAugment-augmented log-mel spectrograms on GPU. This replaces the CPU-side `ASTFeatureExtractor` call that previously happened inside each DataLoader worker.

The key change in the inner loop:
```
# Old (Notebook 1): mel computed per sample in DataLoader worker on CPU
input_values = batch['input_values'].to(DEVICE)

# New (this notebook): raw waveforms arrive, mel computed once per batch on GPU  
waveforms    = batch['waveform'].to(DEVICE)
input_values = compute_log_mel(waveforms, training=True)
```

Everything else — gradient accumulation, gradient clipping, cosine schedule — is identical.

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    f1_metric.reset()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc='Training')):
        waveforms = batch['waveform'].to(DEVICE)
        labels    = batch['label'].to(DEVICE)

        input_values = compute_log_mel(waveforms, training=True)

        outputs = model(input_values=input_values)
        loss    = criterion(outputs.logits, labels) / GRAD_ACCUM
        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM
        preds = torch.argmax(outputs.logits, dim=1)
        f1_metric.update(preds, labels)

    return total_loss / len(loader), f1_metric.compute().item()


def validate(model, loader):
    model.eval()
    f1_metric.reset()
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation'):
            waveforms    = batch['waveform'].to(DEVICE)
            labels       = batch['label'].to(DEVICE)
            input_values = compute_log_mel(waveforms, training=False)
            outputs      = model(input_values=input_values)
            preds        = torch.argmax(outputs.logits, dim=1)
            f1_metric.update(preds, labels)
    return f1_metric.compute().item()

print("train_one_epoch and validate functions ready.")

## 15. Phase 1 — Classifier Head Warm-Up

Identical strategy to Notebook 1: freeze the backbone, train only the 10-class head for `EPOCHS_P1` epochs at `LR_P1 = 3e-4`. This prevents the randomly-initialised head from corrupting the AudioSet backbone weights with large gradient updates.

In [ ]:
for param in model.base_model.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1: {trainable:,} trainable parameters (classifier head only)")

optimizer_p1   = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_P1, weight_decay=WEIGHT_DECAY
)
total_steps_p1 = (len(train_loader) // GRAD_ACCUM) * EPOCHS_P1
scheduler_p1   = get_cosine_schedule_with_warmup(
    optimizer_p1,
    num_warmup_steps   = total_steps_p1 // 10,
    num_training_steps = total_steps_p1
)

best_p1_f1   = 0
patience_cnt = 0

print(f"\n{'='*55}")
print(f"PHASE 1 — Classifier Warm-Up  ({EPOCHS_P1} epochs max)")
print(f"{'='*55}\n")

for epoch in range(EPOCHS_P1):
    train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer_p1, scheduler_p1)
    val_f1 = validate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS_P1}")
    print(f"  Train Loss : {train_loss:.4f}")
    print(f"  Train F1   : {train_f1:.4f}")
    print(f"  Val F1     : {val_f1:.4f}")

    if val_f1 > best_p1_f1:
        best_p1_f1 = val_f1
        torch.save(model.state_dict(), '/kaggle/working/best_model_ta_phase1.pth')
        patience_cnt = 0
        print(f"  Saved best Phase 1 model  (val F1: {best_p1_f1:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE_P1:
            print("  Early stopping Phase 1.")
            break

print(f"\nPhase 1 complete. Best val F1: {best_p1_f1:.4f}")

## 16. Phase 2 — Full Fine-Tuning

Load the best Phase 1 checkpoint, unfreeze all layers, and fine-tune end-to-end with asymmetric learning rates: backbone at `LR_P2 = 1e-5`, classifier head at `LR_P2 × 10 = 1e-4`. The cosine schedule with warm-up is the same as Notebook 1.

In [ ]:
model.load_state_dict(torch.load('/kaggle/working/best_model_ta_phase1.pth'))

for param in model.parameters():
    param.requires_grad = True

print(f"All {sum(p.numel() for p in model.parameters()):,} parameters unfrozen.")

optimizer_p2 = torch.optim.AdamW([
    {'params': model.base_model.parameters(),  'lr': LR_P2},
    {'params': model.classifier.parameters(),  'lr': LR_P2 * LR_HEAD_MULT},
], weight_decay=WEIGHT_DECAY)

total_steps_p2 = (len(train_loader) // GRAD_ACCUM) * EPOCHS_P2
scheduler_p2   = get_cosine_schedule_with_warmup(
    optimizer_p2,
    num_warmup_steps   = total_steps_p2 // 10,
    num_training_steps = total_steps_p2
)

best_p2_f1   = 0
patience_cnt = 0

print(f"\n{'='*55}")
print(f"PHASE 2 — Full Fine-Tune  ({EPOCHS_P2} epochs max)")
print(f"Base LR: {LR_P2}  |  Head LR: {LR_P2 * LR_HEAD_MULT}")
print(f"{'='*55}\n")

for epoch in range(EPOCHS_P2):
    train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer_p2, scheduler_p2)
    val_f1 = validate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS_P2}")
    print(f"  Train Loss : {train_loss:.4f}")
    print(f"  Train F1   : {train_f1:.4f}")
    print(f"  Val F1     : {val_f1:.4f}")

    if val_f1 > best_p2_f1:
        best_p2_f1 = val_f1
        torch.save(model.state_dict(), '/kaggle/working/best_model_ta_phase2.pth')
        patience_cnt = 0
        print(f"  Saved best Phase 2 model  (val F1: {best_p2_f1:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE_P2:
            print("  Early stopping Phase 2.")
            break

print(f"\nPhase 2 complete. Best val F1: {best_p2_f1:.4f}")

## 17. Test-Time Augmentation (TTA)

Same strategy as Notebook 1: take `N_TTA = 7` independent random crops, average the softmax probabilities, return the argmax. The difference here is that the mel spectrogram for each TTA crop is computed on GPU via `compute_log_mel`, not via `ASTFeatureExtractor` on CPU.

In [ ]:
def predict_with_tta(model, waveform_1d, n_tta=N_TTA):
    """
    Args:
        waveform_1d: 1D numpy array or tensor, full-length audio
        n_tta: number of random crops to average over
    Returns:
        int — predicted genre id
    """
    if isinstance(waveform_1d, np.ndarray):
        waveform_1d = torch.from_numpy(waveform_1d)

    all_probs = []
    for _ in range(n_tta):
        wav     = waveform_1d.unsqueeze(0)             # (1, T)
        cropped = crop_random_ta(wav).squeeze(0)        # (AUDIO_LEN,)
        cropped = normalize_ta(cropped.unsqueeze(0)).squeeze(0)
        batch   = cropped.unsqueeze(0).to(DEVICE)       # (1, AUDIO_LEN)
        inp     = compute_log_mel(batch, training=False) # (1, 1024, 128)

        with torch.no_grad():
            outputs = model(input_values=inp)
            probs   = torch.softmax(outputs.logits, dim=1)
            all_probs.append(probs.cpu().numpy())

    avg_probs = np.mean(all_probs, axis=0)
    return int(np.argmax(avg_probs, axis=1)[0])

print(f"TTA function ready. Using {N_TTA} crops per prediction.")

## 18. Generate Submission

In [ ]:
model.load_state_dict(torch.load('/kaggle/working/best_model_ta_phase2.pth'))
model.eval()
print(f"Best Phase 2 model loaded. Val F1: {best_p2_f1:.4f}")

test_df   = pd.read_csv(os.path.join(BASE_PATH, 'test.csv'))
print(f"Test samples: {len(test_df)}")

all_ids, all_preds = [], []

for idx in tqdm(range(len(test_df)), desc=f'TTA Inference (x{N_TTA})'):
    row      = test_df.iloc[idx]
    wav, sr  = torchaudio.load(os.path.join(BASE_PATH, row['filename']))
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        wav = AF.resample(wav, orig_freq=sr, new_freq=SAMPLE_RATE)
    pred = predict_with_tta(model, wav.squeeze(0), n_tta=N_TTA)
    all_ids.append(row['id'])
    all_preds.append(id2label[pred])

submission = pd.DataFrame({'id': all_ids, 'genre': all_preds})
submission.to_csv('/kaggle/working/submission_torchaudio.csv', index=False)

print(f"\nSubmission saved: /kaggle/working/submission_torchaudio.csv")
print(f"Total predictions: {len(submission)}")
print(f"\nPredicted genre distribution:")
print(submission['genre'].value_counts().to_string())

## 19. Training Summary

In [ ]:
print('=' * 60)
print('TRAINING COMPLETE — TorchAudio GPU Pipeline')
print('=' * 60)
print(f'  Phase 1 best val F1 : {best_p1_f1:.4f}')
print(f'  Phase 2 best val F1 : {best_p2_f1:.4f}')
print(f'  TTA crops used      : {N_TTA}')
print(f'  Expected Kaggle F1  : ~{best_p2_f1 + 0.02:.4f}')
print('=' * 60)
print()
